In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import base64
import mimetypes
import pandas as pd
import re
import json
import os

In [ ]:
"""
An Excel file is expected with the following columns:
compound, sentence, image1_name, image1_name_path, image1_caption, image2_name, image2_name_path, image2_caption, image3_name, image3_name_path, image3_caption, image4_name, image4_name_path, image4_caption, image5_name, image5_name_path, image5_caption.

The compound column contains the PIE, and the sentence column contains the context sentence in which the PIE appears. For each image x, imagex_name stores the image filename, imagex_name_path stores the path to the image, and imagex_caption provides a textual description of the image.
"""
df = pd.read_excel("resolved_image_paths.xlsx")
df

In [ ]:
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

In [ ]:
MODEL_TYPE = "GEMINI"  # GEMINI or GPT
#model = ChatOpenAI(model="o3-2025-04-16")

model = ChatGoogleGenerativeAI(
    model="gemini-3-pro-preview",
    api_key="GEMINI_API_KEY"
)

In [ ]:
def extract_json_response(res):
    pattern = r'\{\s*"hasIdiom"\s*:\s*\d+\s*,\s*"explanation"\s*:\s*"([^"\\]*(\\.[^"\\]*)*)"\s*\}'
    pattern = re.compile(
    r'\{\s*'
    r'"?hasIdiom"?\s*:\s*(0|1)\s*,\s*'
    r'"?explanation"?\s*:\s*"?(?P<explanation>.*?)"?\s*,\s*'
    r'"?image_ranks"?\s*:\s*\[\s*'
    r'("?(img[1-5])"?\s*,\s*){4}'
    r'"?(img[1-5])"?'
    r'\s*\]\s*'
    r'\}',
    re.DOTALL
    )
    match = re.search(pattern, res)
    result = None
    if match:
        try:
            json_str = match.group()
            result = json.loads(json_str)
        except:
            result = None
    return result

def encode_image_to_data_url(image_path):
    mime, _ = mimetypes.guess_type(image_path)
    if mime is None:
        mime = "image/png"  # fallback

    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode()

    return f"data:{mime};base64,{data}"

def build_multimodal_message(compound, sentence, image_paths, captions):
    """
    images: dict like {"img1": url_or_base64, ..., "img5": url_or_base64}
    captions: dict like {"caption1": "...", ..., "caption5": "..."}
    """

    blocks = []

    blocks.append({
        "type": "text",
        "text": f"""
            You are a professional linguist specializing in the interpretation of potentially idiomatic expressions (PIEs). You will be given:
            - A PIE
            - A context sentence containing that PIE
            - Five images and their captions.
            Your tasks are:
            1. Interpret the meaning of the PIE within the given sentence.
            2. Determine what semantic contribution the PIE makes in context.
            3. Decide whether the PIE is used literally or idiomatically.
            4. Output 1 for idiomatic, 0 for literal.
            5. Evaluate each image and caption.
            6. Determine how well each image illustrates the intended meaning of the PIE in this sentence.
            7. Rank the images from best match to worst match based on how well they convey the contextual meaning.
            8. Return your output only in the following strict JSON format with following fileds:
                  "hasIdiom": 1 or 0,
                  "explanation": "Concise explanation of your reasoning.",
                  "image_ranks": ["imgX", "imgY", "imgZ", "imgW", "imgV"]

            hasIdiom: 1 if the PIE is used idiomatically, 0 if literal.
            image_ranks: A list of the five image identifiers ordered from best match to worst. Example: ["img2", "img3", "img4", "img1", "img5"]

            The context sentence for PIE: {compound} is given below:
            "{sentence}"
        """
    })

    for i in range(1, 6):
        ident = f"img{i}"
        cap = f"caption{i}"
        data_url = encode_image_to_data_url(image_paths[ident])
        blocks.append({"type": "text", "text": ident})
        blocks.append(
            {"type": "image_url", "image_url": {"url": data_url}}
        )
        blocks.append({"type": "text", "text": captions[cap]})

    return blocks

In [ ]:
has_idiom = []
explanations = []
image_ranks = []


for index, row in df.iterrows():
    print(index)
    compound = row["compound"]
    sentence = row["sentence"]

    image_paths = {
        "img1": row["image1_name_path"],
        "img2": row["image2_name_path"],
        "img3": row["image3_name_path"],
        "img4": row["image4_name_path"],
        "img5": row["image5_name_path"],
    }

    captions = {
        "caption1": row["image1_caption"],
        "caption2": row["image2_caption"],
        "caption3": row["image3_caption"],
        "caption4": row["image4_caption"],
        "caption5": row["image5_caption"],
    }

    message = HumanMessage(
        content=build_multimodal_message(compound, sentence, image_paths, captions)
    )

    resp = model.invoke([message]).content

    if (MODEL_TYPE== "GPT"):
        res_dict = extract_json_response(resp)
    elif(MODEL_TYPE== "GEMINI"):
        res_dict = extract_json_response(resp[0]["text"])


    if res_dict == None:
        has_idiom.append("Wrong formatting")
        explanations.append("Wrong formatting")
        image_ranks.append("Wrong formatting")
        print("Wrong formatting")
        continue
    if res_dict["hasIdiom"] == 1:
        has_idiom.append("idiomatic")
    elif res_dict["hasIdiom"] == 0:
        has_idiom.append("literal")
    explanations.append(res_dict["explanation"])
    image_ranks.append(res_dict["image_ranks"])
    print(res_dict)

In [ ]:
df["hasIdiom"] = has_idiom
df["explanation"] = explanations
df["pred_order"] = image_ranks
df.to_excel("results_1step.xlsx")